In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    LongType,
    ByteType,
)
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from typing import Union
from pyspark.sql.window import Window
from functools import reduce

In [0]:
catalog_name = "databricks-repo"

schema_name = "gold"
schema_silver = 'silver'

default_schema = f'`{catalog_name}`.`{schema_name}`'
silver_schema = f'`{catalog_name}`.`{schema_silver}`'

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog_name}`.`{schema_name}`")

In [0]:
@F.udf(returnType=IntegerType())
def one_if(
    value: Union[StringType, IntegerType],
    compare_to: int | str,
    handle_null_as: Union[StringType, IntegerType],
):
    return F.when(value == compare_to, 1).otherwise(0)

In [0]:
# Load main tables
participantes = spark.table(f"{silver_schema}.`participantes`")
unidade_federativa = spark.table(f"{silver_schema}.`unidade_federativa`")
municipio = spark.table(f"{silver_schema}.`municipio`")
escola = spark.table(f"{silver_schema}.`escola`")

# Load Features tables
ft_faixa_etaria = spark.table(f"{silver_schema}.`ft_faixa_etaria`")
ft_estado_civil = spark.table(f"{silver_schema}.`ft_estado_civil`")
ft_cor_raca = spark.table(f"{silver_schema}.`ft_cor_raca`")
ft_nacionalidade = spark.table(f"{silver_schema}.`ft_nacionalidade`")
ft_conclusao = spark.table(f"{silver_schema}.`ft_conclusao`")
ft_ensino = spark.table(f"{silver_schema}.`ft_ensino`")

# Rename Features tables
ft_faixa_etaria = ft_faixa_etaria.withColumnRenamed("valor_categoria", "faixa_etaria")
ft_estado_civil = ft_estado_civil.withColumnRenamed("valor_categoria", "estado_civil")
ft_cor_raca = ft_cor_raca.withColumnRenamed("valor_categoria", "cor_raca")
ft_nacionalidade = ft_nacionalidade.withColumnRenamed(
    "valor_categoria", "nacionalidade"
)
ft_conclusao = ft_conclusao.withColumnRenamed("valor_categoria", "conclusao")
ft_ensino = ft_ensino.withColumnRenamed("valor_categoria", "tipo_ensino")

In [0]:
dim_participante = (
    participantes.join(
        ft_faixa_etaria,
        participantes.id_faixa_etaria == ft_faixa_etaria.id_categoria,
        "left",
    )
    .join(
        ft_estado_civil,
        participantes.id_estado_civil == ft_estado_civil.id_categoria,
        "left",
    )
    .join(ft_cor_raca, participantes.id_cor_raca == ft_cor_raca.id_categoria, "left")
    .join(
        ft_nacionalidade,
        participantes.id_nacionalidade == ft_nacionalidade.id_categoria,
        "left",
    )
    .join(ft_conclusao, participantes.st_conclusao == ft_conclusao.id_categoria, "left")
    .join(ft_ensino, participantes.id_ensino == ft_ensino.id_categoria, "left")
    .join(unidade_federativa, participantes.id_uf == unidade_federativa.id_uf, "left")
    .join(municipio, participantes.id_municipio == municipio.id_muninicipio, "left")
    .join(
        escola,
        (participantes.id_municipio == escola.id_municipio)
        & (participantes.id_uf == escola.id_uf),
        "left",
    )
    .select(
        participantes.id_inscricao,
        participantes.ano,
        participantes.sexo,
        participantes.treineiro,
        participantes.ano_concluiu,
        ft_faixa_etaria.faixa_etaria,
        ft_estado_civil.estado_civil,
        ft_cor_raca.cor_raca,
        ft_nacionalidade.nacionalidade,
        ft_conclusao.conclusao,
        ft_ensino.tipo_ensino,
        unidade_federativa.uf_sigla,
        municipio.muninicipio.alias("municipio"),
        escola.id_escola,
        escola.tipo_escola,
        escola.situacao_funcional,
        participantes.resp_Q001.alias("escolaridade_pai"),
        participantes.resp_Q002.alias("escolaridade_mae"),
        participantes.resp_Q003.alias("ocupacao_mae"),
        participantes.resp_Q004.alias("ocupacao_pai"),
        participantes.resp_Q005.alias("membros_familia"),
        participantes.resp_Q006.alias("renda_familiar"),
        participantes.resp_Q007.alias("situacao_trabalho"),
        participantes.resp_Q008.alias("tipo_moradia"),
    )
)


dim_participante.show(10)
dim_participante.write.mode("overwrite").saveAsTable(
    f"{default_schema}.dim_participante_completo"
)




In [0]:
agg_regional = (
    participantes.join(
        unidade_federativa, participantes.id_uf == unidade_federativa.id_uf, "left"
    )
    .join(municipio, participantes.id_municipio == municipio.id_muninicipio)
    .join(
        escola,
        (participantes.id_municipio == escola.id_municipio)
        & (participantes.id_uf == escola.id_uf),
        "left",
    )
    .groupBy(participantes.ano, unidade_federativa.uf_sigla, municipio.muninicipio)
    .agg(
        F.countDistinct(participantes.id_inscricao).alias("total_participantes"),
        F.sum(F.when(participantes.treineiro == 1, 1).otherwise(0)).alias(
            "total_treineiros"
        ),
        F.sum(F.when(participantes.treineiro == 0, 1).otherwise(0)).alias(
            "total_candidatos_efetivos"
        ),
        F.sum(F.when(participantes.sexo == "M", 1).otherwise(0)).alias(
            "total_masculino"
        ),
        F.sum(F.when(participantes.sexo == "F", 1).otherwise(0)).alias(
            "total_feminino"
        ),
        F.round(
            F.sum(F.when(participantes.sexo == "F", 1).otherwise(0))
            * 100.0
            / F.count("*"),
            2,
        ).alias("pct_feminino"),
        F.sum(
            F.when(participantes.id_faixa_etaria.isin([1, 2, 3]), 1).otherwise(0)
        ).alias("total_ate_20_anos"),
        F.sum(
            F.when(participantes.id_faixa_etaria.isin([4, 5, 6, 7]), 1).otherwise(0)
        ).alias("total_21_30_anos"),
        F.sum(F.when(participantes.id_faixa_etaria >= 8, 1).otherwise(0)).alias(
            "total_mais_30_anos"
        ),
        F.sum(F.when(participantes.id_cor_raca == 1, 1).otherwise(0)).alias(
            "total_branca"
        ),
        F.sum(F.when(participantes.id_cor_raca == 2, 1).otherwise(0)).alias(
            "total_preta"
        ),
        F.sum(F.when(participantes.id_cor_raca == 3, 1).otherwise(0)).alias(
            "total_parda"
        ),
        F.sum(F.when(participantes.id_cor_raca.isin([4, 5, 6]), 1).otherwise(0)).alias(
            "total_outras_racas"
        ),
        F.countDistinct(escola.id_escola).alias("total_escolas"),
        F.sum(F.when(escola.tipo_escola == 1, 1).otherwise(0)).alias(
            "total_escola_publica"
        ),
        F.sum(F.when(escola.tipo_escola == 2, 1).otherwise(0)).alias(
            "total_escola_privada"
        ),
        F.sum(F.when(participantes.st_conclusao == 1, 1).otherwise(0)).alias(
            "total_ja_concluiu"
        ),
        F.sum(F.when(participantes.st_conclusao == 2, 1).otherwise(0)).alias(
            "total_concluindo"
        ),
        F.sum(F.when(participantes.st_conclusao == 3, 1).otherwise(0)).alias(
            "total_nao_concluiu"
        ),
    )
)

agg_regional.show(10)

agg_regional.write.mode("overwrite").saveAsTable(
    f"{default_schema}.agg_participacao_regional"
)



In [0]:
# Join and aggregate
agg_socioeconomico = (
    participantes.join(
        unidade_federativa, participantes.id_uf == unidade_federativa.id_uf, "left"
    )
    .join(ft_cor_raca, participantes.id_cor_raca == ft_cor_raca.id_categoria, "left")
    .join(
        escola,
        (participantes.id_municipio == escola.id_municipio)
        & (participantes.id_uf == escola.id_uf),
        "left",
    )
    .groupBy(
        participantes.ano,
        unidade_federativa.uf_sigla,
        ft_cor_raca.cor_raca,
        participantes.resp_Q006.alias("faixa_renda_familiar"),
    )
    .agg(
        F.countDistinct(participantes.id_inscricao).alias("total_participantes"),
        F.sum(F.when(participantes.resp_Q007 == "A", 1).otherwise(0)).alias(
            "total_nao_trabalha"
        ),
        F.sum(
            F.when(participantes.resp_Q007.isin(["B", "C", "D"]), 1).otherwise(0)
        ).alias("total_trabalha"),
        F.round(
            F.sum(F.when(participantes.resp_Q007.isin(["B", "C", "D"]), 1).otherwise(0))
            * 100.0
            / F.count("*"),
            2,
        ).alias("pct_trabalha"),
        F.sum(F.when(participantes.resp_Q008 == "A", 1).otherwise(0)).alias(
            "total_casa_propria"
        ),
        F.sum(F.when(participantes.resp_Q008.isin(["B", "C"]), 1).otherwise(0)).alias(
            "total_alugada_cedida"
        ),
        F.sum(F.when(participantes.resp_Q014 == "A", 1).otherwise(0)).alias(
            "total_tem_internet"
        ),
        F.sum(F.when(participantes.resp_Q015 == "A", 1).otherwise(0)).alias(
            "total_tem_computador"
        ),
        F.sum(F.when(escola.tipo_escola == 1, 1).otherwise(0)).alias(
            "total_escola_publica"
        ),
        F.sum(F.when(escola.tipo_escola == 2, 1).otherwise(0)).alias(
            "total_escola_privada"
        ),
        F.round(
            F.sum(F.when(escola.tipo_escola == 2, 1).otherwise(0))
            * 100.0
            / F.count("*"),
            2,
        ).alias("pct_escola_privada"),
    )
)


agg_socioeconomico.show(10)

agg_socioeconomico.write.mode("overwrite").partitionBy(['cor_raca','uf_sigla']).saveAsTable(
    f"{default_schema}.agg_perfil_socioeconomico"
)

In [0]:
agg_tendencias = (
    participantes.join(
        escola,
        (participantes.id_municipio == escola.id_municipio)
        & (participantes.id_uf == escola.id_uf),
        "left",
    )
    .groupBy(participantes.ano)
    .agg(
        F.countDistinct(participantes.id_inscricao).alias("total_inscricoes"),
        F.countDistinct(
            F.when(participantes.treineiro == 0, participantes.id_inscricao)
        ).alias("total_efetivos"),
        F.countDistinct(
            F.when(participantes.treineiro == 1, participantes.id_inscricao)
        ).alias("total_treineiros"),
        F.round(
            F.countDistinct(
                F.when(participantes.treineiro == 1, participantes.id_inscricao)
            )
            * 100.0
            / F.countDistinct(participantes.id_inscricao),
            2,
        ).alias("pct_treineiros"),
        F.round(
            F.avg(F.when(participantes.sexo == "F", 1.0).otherwise(0.0)) * 100, 2
        ).alias("pct_feminino"),
        F.round(
            F.avg(F.when(participantes.sexo == "M", 1.0).otherwise(0.0)) * 100, 2
        ).alias("pct_masculino"),
        F.round(
            F.avg(F.when(participantes.id_faixa_etaria <= 3, 1.0).otherwise(0.0)) * 100,
            2,
        ).alias("pct_ate_20_anos"),
        F.round(
            F.avg(
                F.when(participantes.id_faixa_etaria.between(4, 7), 1.0).otherwise(0.0)
            )
            * 100,
            2,
        ).alias("pct_21_30_anos"),
        F.round(
            F.avg(F.when(participantes.id_faixa_etaria >= 8, 1.0).otherwise(0.0)) * 100,
            2,
        ).alias("pct_mais_30_anos"),
        F.round(
            F.avg(F.when(participantes.id_cor_raca == 1, 1.0).otherwise(0.0)) * 100, 2
        ).alias("pct_branca"),
        F.round(
            F.avg(F.when(participantes.id_cor_raca == 2, 1.0).otherwise(0.0)) * 100, 2
        ).alias("pct_preta"),
        F.round(
            F.avg(F.when(participantes.id_cor_raca == 3, 1.0).otherwise(0.0)) * 100, 2
        ).alias("pct_parda"),
        F.round(
            F.avg(F.when(participantes.id_cor_raca == 4, 1.0).otherwise(0.0)) * 100, 2
        ).alias("pct_amarela"),
        F.round(
            F.avg(F.when(participantes.id_cor_raca == 5, 1.0).otherwise(0.0)) * 100, 2
        ).alias("pct_indigena"),
        F.round(
            F.avg(F.when(participantes.st_conclusao == 1, 1.0).otherwise(0.0)) * 100, 2
        ).alias("pct_ja_concluiu"),
        F.round(
            F.avg(F.when(participantes.st_conclusao == 2, 1.0).otherwise(0.0)) * 100, 2
        ).alias("pct_concluindo"),
        F.round(
            F.avg(F.when(participantes.st_conclusao == 3, 1.0).otherwise(0.0)) * 100, 2
        ).alias("pct_nao_concluiu"),
        F.countDistinct(escola.id_escola).alias("total_escolas_participantes"),
        F.round(
            F.avg(F.when(escola.tipo_escola == 1, 1.0).otherwise(0.0)) * 100, 2
        ).alias("pct_escola_publica"),
        F.round(
            F.avg(F.when(escola.tipo_escola == 2, 1.0).otherwise(0.0)) * 100, 2
        ).alias("pct_escola_privada"),
        F.countDistinct(participantes.id_uf).alias("total_ufs_participantes"),
        F.countDistinct(participantes.id_municipio).alias(
            "total_municipios_participantes"
        ),
        F.round(
            F.avg(
                F.when(participantes.resp_Q007.isin(["B", "C", "D"]), 1.0).otherwise(
                    0.0
                )
            )
            * 100,
            2,
        ).alias("pct_trabalha"),
        F.round(
            F.avg(F.when(participantes.resp_Q014 == "A", 1.0).otherwise(0.0)) * 100, 2
        ).alias("pct_tem_internet"),
        F.round(
            F.avg(F.when(participantes.resp_Q015 == "A", 1.0).otherwise(0.0)) * 100, 2
        ).alias("pct_tem_computador"),
    )
)

agg_tendencias.write.mode("overwrite").saveAsTable(f"{default_schema}.agg_tendencias_anuais")
agg_tendencias.show(10)